In [ ]:
# Getting Started with GX Core

We will learn how to:

- connect to data
- create expectations
- validate data
- review validation results

First of all, let's import the core library of Great Expectations.

Specifically, the `great_expectations` module is the root of the GX Core library.</br>
It contains shortcuts and convenience methods for starting a GX project in a Python session.


In [ ]:
import great_expectations as gx
import pandas as pd

## Importing the data

For this lab, we will be using a sample of [NYC taxi trip record data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

> Yellow and green **taxi trip records** include fields capturing _pick-up and drop-off dates/times_, _pick-up and drop-off locations_, _trip distances_, _itemized fares_, _rate types_, _payment types_, and driver-reported _passenger counts_. The data used in the attached datasets were collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers authorized under the Taxicab & Livery Passenger Enhancement Programs (TPEP/LPEP).

Let's download and read the sample data into a Pandas DataFrame.


In [ ]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/great-expectations/gx_tutorials/main/data/yellow_tripdata_sample_2019-01.csv"
)
df.head()

### Create a **Data Context**

A **Data Context** object serves as the entrypoint for interacting with GX components.


In [ ]:
context = gx.get_context()

### Connect to data and create a **Batch**

Let's define:

- a **Data Source**
- a **Data Asset**
- a **Batch Definition**
- a **Batch**


In [ ]:
data_source = context.data_sources.add_pandas("pandas")
data_asset = data_source.add_dataframe_asset(name="pd dataframe asset")

batch_definition = data_asset.add_batch_definition_whole_dataframe("batch definition")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df})

### Create an **Expectation**

We expect that the contents of the column `passenger_count` consist of values ranging from 1 to 6.


In [ ]:
expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="passenger_count", min_value=1, max_value=6
)

### Validate the data batch

Using the expectation you just created, validate the data batch.


In [ ]:
validation_result = batch.validate(expectation)

In [ ]:
print(validation_result)

### Create an **Expectation Suite**


In [ ]:
suite = context.suites.add(
    gx.ExpectationSuite(
        name="Taxi trip expectations",
    )
)

In [ ]:
# Expectation 1
suite.add_expectation(expectation)

# Expectation 2
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="fare_amount", min_value=0)
)

# Expectation 3
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="pickup_datetime")
)

# Expectation 4
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(column="payment_type", type_="int")
)

# Expectation 5
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="payment_type")
)

# Expectation 6
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="payment_type", value_set=[1, 2, 3, 4]
    )
)


### Create a **Validation Definition**

A **Validation Definition** explicitly ties together the Batch of data to be validated to the Expectation Suite that should be used to validate it.


In [ ]:
validation_definition = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="Validation definition", data=batch_definition, suite=suite
    )
)

### Create and run a **Checkpoint**

Create and run a Checkpoint to validate the data based on the supplied Validation Definition.


In [ ]:
checkpoint = context.checkpoints.add(
    gx.Checkpoint(
        name="checkpoint", validation_definitions=[validation_definition]
    )
)
checkpoint_result = checkpoint.run(batch_parameters={"dataframe": df})
print(checkpoint_result.describe())


---

## References

[1] https://docs.greatexpectations.io/docs/core/introduction/try_gx/

[2] [Expectations Gallery](https://greatexpectations.io/expectations/)
